# GraphRAG Retrievers

Now that your data is loaded with embeddings, you'll build complete question-answering pipelines using the neo4j-graphrag library's retriever classes with Databricks Model Serving.

**Prerequisites:** Complete [01 Data and Embeddings](01_data_and_embeddings.ipynb) first.

**Learning Objectives:**
- Use `VectorRetriever` for semantic search
- Build a `GraphRAG` pipeline for question answering
- Use `VectorCypherRetriever` to traverse graph relationships for richer context
- Compare standard vs. graph-enhanced retrieval

## Setup

In [ ]:
# Install dependencies for Databricks Model Serving
%pip install neo4j-graphrag databricks-langchain python-dotenv pydantic-settings nest-asyncio -q

In [ ]:
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.generation import GraphRAG

from data_utils import Neo4jConnection, get_llm, get_embedder

In [ ]:
neo4j = Neo4jConnection().verify()
driver = neo4j.driver

llm = get_llm()
embedder = get_embedder()

print(f"LLM: {llm.model_id}")
print(f"Embedder: {embedder.endpoint}")

## VectorRetriever

The `VectorRetriever` handles embedding generation and Neo4j queries automatically.

In [ ]:
vector_retriever = VectorRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    return_properties=['text']
)

# Diagnostic search
query = "What products does Apple make?"
result = vector_retriever.search(query_text=query, top_k=3)

print(f"Query: \"{query}\"")
print(f"Results: {len(result.items)}\n")
for item in result.items:
    score = item.metadata.get('score', 'N/A')
    print(f"Score: {score:.4f} | {str(item.content)[:80]}...")

## GraphRAG Pipeline

The `GraphRAG` class combines retrieval with LLM generation for grounded question answering using Databricks-hosted models.

In [ ]:
rag = GraphRAG(llm=llm, retriever=vector_retriever)

query = "What products does Apple make?"
response = rag.search(query, retriever_config={"top_k": 3})

print(f"Query: \"{query}\"\n")
print("Answer:")
print(response.answer)

## VectorCypherRetriever: Graph-Enhanced Retrieval

The real power of GraphRAG comes from combining vector search with graph traversal. `VectorCypherRetriever` runs a custom Cypher query after finding matches, letting you include adjacent chunks for expanded context.

The `node` variable refers to each chunk found by vector search.

In [ ]:
# Expanded context: concatenate prev + current + next chunks
expanded_context_query = """
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (prev:Chunk)-[:NEXT_CHUNK]->(node)
OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(next:Chunk)
WITH node, doc, prev, next
RETURN 
    COALESCE(prev.text + ' ', '') + node.text + COALESCE(' ' + next.text, '') AS expanded_context,
    doc.path AS source_document,
    node.index AS center_chunk_index
"""

cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    retrieval_query=expanded_context_query
)

print("VectorCypherRetriever initialized with expanded context query")

In [ ]:
query = "What products and services does Apple offer?"

rag_expanded = GraphRAG(llm=llm, retriever=cypher_retriever)
response = rag_expanded.search(query, retriever_config={"top_k": 2}, return_context=True)

print(f"Query: \"{query}\"\n")
print("Answer:")
print(response.answer)

In [ ]:
# View retrieved context (includes adjacent chunks)
print("\nRetrieved Context:")
print("=" * 60)
for i, item in enumerate(response.retriever_result.items):
    print(f"\n[Result {i+1}]")
    print(item.content)

## Comparison: Standard vs. Expanded Context

The expanded context often produces more complete answers because the LLM has surrounding information.

In [ ]:
query = "What does Apple do?"

# Standard retriever
print("=== Standard VectorRetriever ===")
rag_standard = GraphRAG(llm=llm, retriever=vector_retriever)
response_standard = rag_standard.search(query, retriever_config={"top_k": 2})
print(response_standard.answer)

# Expanded context retriever
print("\n=== VectorCypherRetriever (Expanded Context) ===")
response_expanded = rag_expanded.search(query, retriever_config={"top_k": 2})
print(response_expanded.answer)

## Summary

You've built complete GraphRAG pipelines using Databricks Model Serving:

1. **VectorRetriever** - Simple semantic search with automatic embedding
2. **GraphRAG** - Retrieval + LLM generation for grounded answers
3. **VectorCypherRetriever** - Custom Cypher queries for graph-enhanced retrieval
4. **Expanded context pattern** - Include adjacent chunks via `NEXT_CHUNK` traversal

**Key insight:** Vector search finds the needle; graph traversal provides the context around it.

---

**Congratulations!** You've completed the GraphRAG lab.

**Next:** [Lab 6 - Aura Agents API](../Lab_6_Aura_Agents_API/README.md)

In [ ]:
neo4j.close()